# Experiment 1: Pascal VOC 5-Seed Training

**Purpose:** Primary benchmark (Table 1 in manuscript). Trains both `quantized` and `standard` variants with 5 seeds for mean ± std reporting.

**Dataset:** Pascal VOC 2007+2012 (16.5K train / 4.9K test)

**Expected time:** ~2-3 hours per seed on T4 at 416×416 (~20-30 hours total)

**Runtime:** GPU (T4 or better recommended)

> ⚠️ **Tip:** If short on time, run seeds 42 and 123 first, then continue with 256/512/1024 in another session.

In [ ]:
# === Cell 1: Setup ===
!rm -rf /content/tinyYOLO 2>/dev/null
!git clone https://github.com/ShMazumder/tinyYOLO.git /content/tinyYOLO
%cd /content/tinyYOLO
!pip install -e . -q
!pip install tqdm -q

# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
# === Cell 2: Download VOC Dataset ===
!python -c "from ultralytics.data.utils import check_det_dataset; check_det_dataset('VOC.yaml')"

In [ ]:
# === Cell 3: Configuration ===
SEEDS = [42, 123, 256, 512, 1024]
VARIANTS = ['quantized', 'standard']
IMGSZ = 416
EPOCHS = 300
WARMUP = 3
DATA = 'voc.yaml'

print(f"Plan: {len(SEEDS)} seeds × {len(VARIANTS)} variants = {len(SEEDS)*len(VARIANTS)} runs")
print(f"Settings: {IMGSZ}×{IMGSZ}, {EPOCHS} epochs, warmup={WARMUP}")
print(f"Seeds: {SEEDS}")
print(f"Estimated time: ~{len(SEEDS)*len(VARIANTS)*2.5:.0f} hours on T4")

In [ ]:
# === Cell 4: Train QUANTIZED variant — all 5 seeds ===
# Uses get_ipython().system() to preserve tqdm \r overwrites in Colab
for i, seed in enumerate(SEEDS):
    name = f"voc-q-{IMGSZ}-seed{seed}"
    print(f"\n{'='*70}")
    print(f"  [{i+1}/{len(SEEDS)}] Quantized seed={seed}")
    print(f"{'='*70}", flush=True)
    get_ipython().system(
        f"python scripts/train.py --task det --variant quantized "
        f"--data {DATA} --imgsz {IMGSZ} --epochs {EPOCHS} "
        f"--seed {seed} --warmup {WARMUP} --name {name}"
    )
print("\n🎉 All quantized seeds complete!")

In [ ]:
# === Cell 5: Train STANDARD variant — all 5 seeds ===
for i, seed in enumerate(SEEDS):
    name = f"voc-std-{IMGSZ}-seed{seed}"
    print(f"\n{'='*70}")
    print(f"  [{i+1}/{len(SEEDS)}] Standard seed={seed}")
    print(f"{'='*70}", flush=True)
    get_ipython().system(
        f"python scripts/train.py --task det --variant standard "
        f"--data {DATA} --imgsz {IMGSZ} --epochs {EPOCHS} "
        f"--seed {seed} --warmup {WARMUP} --name {name}"
    )
print("\n🎉 All standard seeds complete!")

In [ ]:
# === Cell 6: Collect Results — Mean ± Std ===
import json, glob, numpy as np
from pathlib import Path

print("=" * 80)
print("  Pascal VOC Results — Mean ± Std (5 seeds)")
print("=" * 80)

all_results = {}
for variant in ['q', 'std']:
    label = 'Quantized' if variant == 'q' else 'Standard'
    maps50, maps95, precisions, recalls = [], [], [], []

    for f in sorted(glob.glob(f'experiments/results/voc-{variant}-{IMGSZ}-seed*/config.json')):
        with open(f) as fh:
            cfg = json.load(fh)
        fm = cfg.get('final_metrics', {})
        seed_name = Path(f).parent.name
        m50 = fm.get('mAP50', 0)
        m95 = fm.get('mAP50_95', fm.get('mAP50-95', 0))
        p = fm.get('precision', 0)
        r = fm.get('recall', 0)
        maps50.append(m50); maps95.append(m95)
        precisions.append(p); recalls.append(r)
        print(f"  {seed_name}: mAP@50={m50:.4f}  mAP@50-95={m95:.4f}  P={p:.4f}  R={r:.4f}")

    if maps50:
        print(f"\n  {label} Summary (n={len(maps50)}):")
        print(f"    mAP@50:    {np.mean(maps50)*100:.1f} ± {np.std(maps50)*100:.1f}%")
        print(f"    mAP@50-95: {np.mean(maps95)*100:.1f} ± {np.std(maps95)*100:.1f}%")
        print(f"    Precision: {np.mean(precisions)*100:.1f} ± {np.std(precisions)*100:.1f}%")
        print(f"    Recall:    {np.mean(recalls)*100:.1f} ± {np.std(recalls)*100:.1f}%")
        all_results[variant] = {
            'mAP50': maps50, 'mAP50_95': maps95,
            'precision': precisions, 'recall': recalls
        }
    print()

# Save consolidated results
with open('experiments/results/voc_5seed_summary.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print("Saved: experiments/results/voc_5seed_summary.json")

In [ ]:
# === Cell 7: Generate Publication Plots ===
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.size'] = 12

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: mAP@50 per seed (bar chart)
ax = axes[0]
x = np.arange(len(SEEDS))
width = 0.35
if 'q' in all_results:
    ax.bar(x - width/2, [v*100 for v in all_results['q']['mAP50']], width,
           label='Quantized', color='#2196F3', alpha=0.85)
if 'std' in all_results:
    ax.bar(x + width/2, [v*100 for v in all_results['std']['mAP50']], width,
           label='Standard', color='#FF9800', alpha=0.85)
ax.set_xlabel('Seed'); ax.set_ylabel('mAP@50 (%)')
ax.set_title('VOC mAP@50 — Per Seed'); ax.set_xticks(x, SEEDS)
ax.legend(); ax.grid(axis='y', alpha=0.3)

# Plot 2: Box plot comparison
ax = axes[1]
data_to_plot, labels = [], []
if 'q' in all_results:
    data_to_plot.append([v*100 for v in all_results['q']['mAP50']])
    labels.append('Quantized')
if 'std' in all_results:
    data_to_plot.append([v*100 for v in all_results['std']['mAP50']])
    labels.append('Standard')
bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True, boxprops=dict(alpha=0.7))
colors = ['#2196F3', '#FF9800']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_ylabel('mAP@50 (%)'); ax.set_title('VOC mAP@50 — Distribution')
ax.grid(axis='y', alpha=0.3)

# Plot 3: Training curves overlay
ax = axes[2]
for variant, color in [('q', '#2196F3'), ('std', '#FF9800')]:
    histories = []
    for f in sorted(glob.glob(f'experiments/results/voc-{variant}-{IMGSZ}-seed*/history.json')):
        with open(f) as fh:
            histories.append(json.load(fh))
    if histories:
        min_len = min(len(h) for h in histories)
        mean_map = [np.mean([h[e].get('mAP50', 0) for h in histories]) for e in range(min_len)]
        ax.plot(range(1, min_len+1), mean_map, color=color, linewidth=2.5,
                label=f"{'Quantized' if variant=='q' else 'Standard'} (mean)")
ax.set_xlabel('Epoch'); ax.set_ylabel('mAP@50')
ax.set_title('Training Convergence — All Seeds'); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('TinyYOLO — Pascal VOC 5-Seed Results (416×416, 300 epochs)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('experiments/results/voc_5seed_results.png', dpi=200, bbox_inches='tight')
plt.show()
print("\nSaved: experiments/results/voc_5seed_results.png")

In [ ]:
# === Cell 8: Statistical Significance — t-test ===
from scipy import stats

if 'q' in all_results and 'std' in all_results:
    q_maps = all_results['q']['mAP50']
    s_maps = all_results['std']['mAP50']
    t_stat, p_value = stats.ttest_ind(q_maps, s_maps)
    print(f"Two-sample t-test (Quantized vs Standard):")
    print(f"  t-statistic: {t_stat:.4f}")
    print(f"  p-value:     {p_value:.6f}")
    print(f"  Significant: {'YES (p < 0.01)' if p_value < 0.01 else 'YES (p < 0.05)' if p_value < 0.05 else 'NO'}")
else:
    print("Need both variants to run t-test.")

In [ ]:
# === Cell 9: Download Results ===
!cd experiments && zip -r /content/voc_5seed_results.zip results/voc-*
print("\n📥 Download: /content/voc_5seed_results.zip")
# In Colab, click the Files panel (left sidebar) to download.